# 8장. 양자화: 8비트와 4비트

7장의 모델을 같은 시드로 다시 학습시킨 다음, 가중치를 8비트, 4비트 정수로 직접 바꿔 본다. 가중치의 오차, 검증 손실과 다음 토큰, 어느 부분이 예민한지, 메모리와 속도를 잰다.

**준비:** 한국어 위키백과 덤프의 첫 조각 `kowiki-latest-pages-articles1.xml-p1p82407.bz2`(4장과 같은 파일)를 왼쪽 파일 창에 올리고, 코랩에서는 `!pip install tokenizers`를 먼저 실행한다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 14분쯤 걸렸다.

## 8.1 7장의 모델과 가중치

In [ ]:
import bz2, re, math, time, os, sys, random, json, platform
from collections import Counter
from xml.etree.ElementTree import iterparse
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
os.environ["RAYON_NUM_THREADS"] = "2"
import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
torch.set_num_threads(2)

pages = []
with bz2.open("kowiki-latest-pages-articles1.xml-p1p82407.bz2", "rb") as f:
    for _, el in iterparse(f, events=("end",)):
        if el.tag.endswith("}page"):
            if el.findtext("{*}ns") == "0" and el.find("{*}redirect") is None:
                pages.append((el.findtext("{*}title"), el.findtext("{*}revision/{*}text") or ""))
            el.clear()
print(len(pages))
print(sys.version.split()[0], torch.__version__, tokenizers.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
# 4장의 clean()을 그대로 쓴다. 이번에는 문장으로 나누지 않고 문단을 이어 문서 하나를 글 하나로 만든다
def strip_nested(text, left, right):
    """가장 안쪽의 left ... right를 지우기를 더 지울 것이 없을 때까지 한다(틀 {{ }}, 표 {| |})."""
    inner = re.compile(re.escape(left) + r"(?:(?!" + re.escape(left) + r").)*?" + re.escape(right), re.S)
    while True:
        new = inner.sub("", text)
        if new == text:
            return new
        text = new

def clean(wiki):
    """위키 문법을 걷어 내고 문단(줄) 목록을 돌려준다."""
    t = re.sub(r"<!--.*?-->", "", wiki, flags=re.S)                       # 주석
    t = re.sub(r"<ref[^>]*/>", "", t)                                      # 각주
    t = re.sub(r"<ref[^>]*>.*?</ref>", "", t, flags=re.S)
    t = strip_nested(t, "{{", "}}")                                        # 틀(정보 상자 등)
    t = strip_nested(t, "{|", "|}")                                        # 표
    t = re.sub(r"<(math|gallery|timeline|score|syntaxhighlight|source|pre|code)[^>]*>.*?</\1>", "", t, flags=re.S | re.I)
    t = re.sub(r"<[^>]+>", "", t)                                          # 남은 HTML 태그
    t = re.sub(r"\[\[(파일|File|그림|Image|분류|Category):[^\[\]]*(\[\[[^\]]*\]\][^\[\]]*)*\]\]", "", t, flags=re.I)
    t = re.sub(r"\[\[([^\]|]*)\|([^\]]*)\]\]", r"\2", t)                   # [[문서|보이는 글]] → 보이는 글
    t = re.sub(r"\[\[([^\]]*)\]\]", r"\1", t)                              # [[문서]] → 문서
    t = re.sub(r"\[https?://[^\s\]]+ ([^\]]*)\]", r"\1", t)                # [주소 글] → 글
    t = re.sub(r"\[https?://[^\]]*\]", "", t)
    t = re.sub(r"'{2,}", "", t)                                            # 굵게·기울임
    t = re.sub(r"__[A-Z]+__", "", t)
    out = []
    for line in t.split("\n"):
        line = line.strip()
        if not line or line.startswith(("=", "|", "!", "{", "}")):          # 제목, 표의 나머지
            continue
        line = re.sub(r"^[*#:;]+\s*", "", line)                            # 목록 기호
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            out.append(line)
    return out

def sentences(paragraph):
    return [s.strip() for s in re.split(r"(?<=[.!?。])\s+", paragraph) if s.strip()]

hangul = lambda s: sum("가" <= ch <= "힣" for ch in s)
keep = lambda p: len(p) >= 20 and hangul(p) / max(1, sum(ch.isalpha() for ch in p)) >= 0.5
docs = [" ".join(p for p in clean(text) if keep(p)) for _, text in pages]
held = set(random.Random(0).sample(range(len(pages)), len(pages) // 10))   # 4·5장과 같은 검증용 문서
train_docs = [d for i, d in enumerate(docs) if i not in held and d]
val_docs = [d for i, d in enumerate(docs) if i in held and d]
print(len(train_docs), len(val_docs), sum(map(len, train_docs)), sum(map(len, val_docs)))

In [ ]:
pick = random.Random(0)
paras = [p for d in train_docs for p in d.split(". ")]
letters = {ch for ch, n in Counter(ch for p in pick.sample(paras, 300000) for ch in p).items() if n >= 5}
UNK_CH, SP = "□", "▁"
to_text = lambda t: "".join(ch if ch in letters else UNK_CH for ch in t).replace(" ", SP)
tok = Tokenizer(models.BPE(unk_token=UNK_CH))
tok.pre_tokenizer = pre_tokenizers.Split(SP, behavior="merged_with_next")
tok.train_from_iterator([to_text(p) for p in pick.sample(paras, 200000)], trainers.BpeTrainer(
    vocab_size=8000, show_progress=False, initial_alphabet=sorted(letters - {" "} | {SP, UNK_CH}),
    special_tokens=["<패딩>", "<시작>", "<끝>"]))
PAD, START, END, VOCAB = 0, 1, 2, tok.get_vocab_size()

LENGTH = 512                                               # 모델이 한 번에 읽는 토큰 수
def chunks(texts):
    """문서마다 <시작>·<끝>을 붙여 한 줄로 이은 다음, LENGTH + 1개씩 자른다(마지막 1개는 정답으로만 쓴다)."""
    stream = [t for e in tok.encode_batch([to_text(x) for x in texts]) for t in [START] + e.ids + [END]]
    n = len(stream) // (LENGTH + 1)
    return torch.tensor(stream[:n * (LENGTH + 1)]).view(n, LENGTH + 1)

train_rows = chunks(pick.sample(train_docs, len(train_docs)))
val_rows = chunks(val_docs)
show = lambda ids: tok.decode([i for i in ids if i > END]).replace(" ", "").replace(SP, " ")   # 토큰 번호를 글로
print(VOCAB, len(letters), tuple(train_rows.shape), tuple(val_rows.shape))
print(show(val_rows[0, :40].tolist()))

In [ ]:
def rotate(x, pos):
    """3장 3.2절의 RoPE. 이번에는 위치 번호 pos(배치, 길이)를 받아, 그 위치의 각도만큼 돌린다."""
    hd = x.shape[-1]
    theta = 10000.0 ** (-torch.arange(0, hd, 2).float() / hd)
    angle = pos[:, None, :, None].float() * theta                     # (배치, 1, 길이, hd/2)
    c, s = torch.cos(angle), torch.sin(angle)
    a, b = x[..., 0::2], x[..., 1::2]
    return torch.stack([a * c - b * s, a * s + b * c], dim=-1).flatten(-2)

class Attention(nn.Module):
    """RoPE를 넣은 셀프 어텐션. kv_heads가 heads보다 작으면 그룹 질의 어텐션(3장 3.5절)이다."""
    def __init__(self, dim, heads, kv_heads):
        super().__init__()
        self.heads, self.kv_heads, self.hd = heads, kv_heads, dim // heads
        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(dim, kv_heads * self.hd)
        self.to_v = nn.Linear(dim, kv_heads * self.hd)
        self.out = nn.Linear(dim, dim)

    def forward(self, x, pos, allow, cache=None):
        B, L, D = x.shape
        q = rotate(self.to_q(x).view(B, L, self.heads, self.hd).transpose(1, 2), pos)
        k = rotate(self.to_k(x).view(B, L, self.kv_heads, self.hd).transpose(1, 2), pos)
        v = self.to_v(x).view(B, L, self.kv_heads, self.hd).transpose(1, 2)
        if cache is not None:                                 # 6.2절: 앞에서 만든 키·값 뒤에 이번 것을 붙여 저장한다
            if "k" in cache:
                k, v = torch.cat([cache["k"], k], dim=2), torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=allow, enable_gqa=self.kv_heads < self.heads)
        return self.out(a.transpose(1, 2).reshape(B, L, D))

class Block(nn.Module):
    def __init__(self, dim, heads, kv_heads):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attend = Attention(dim, heads, kv_heads)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x, pos, allow, cache=None):
        x = x + self.attend(self.norm1(x), pos, allow, cache)
        return x + self.mlp(self.norm2(x))

class LanguageModel(nn.Module):
    def __init__(self, dim=192, layers=4, heads=6, kv_heads=6, size=8000):
        super().__init__()
        self.table = nn.Embedding(size, dim)                  # RoPE를 쓰므로 위치 표가 없다
        self.blocks = nn.ModuleList(Block(dim, heads, kv_heads) for _ in range(layers))
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, size)

    def forward(self, ids, pos=None, caches=None, real=None):
        """ids(배치, 길이). caches를 주면 블록마다 키·값을 쌓는다. real은 패딩이 아닌 토큰(6.5절)."""
        B, L = ids.shape
        before = caches[0]["real"].shape[1] if caches and "real" in caches[0] else 0   # 캐시에 이미 있는 토큰 수
        if pos is None:
            pos = torch.arange(before, before + L).expand(B, L)
        if real is None:
            real = torch.ones(B, L, dtype=torch.bool)
        if caches is not None:
            real = torch.cat([caches[0]["real"], real], dim=1) if before else real
            caches[0]["real"] = real
        q_at = torch.arange(before, before + L)                # 질의와 키가 줄에서 몇 번째 칸인지
        k_at = torch.arange(before + L)
        # 앞 칸만 본다(인과 마스크). 패딩 키는 보지 않되, 패딩 칸의 질의는 자기 자신만은 보게 해서 계산이 끊기지 않게 한다
        allow = (k_at[None, :] <= q_at[:, None]) & (real[:, None, :] | (k_at[None, :] == q_at[:, None]))
        x = self.table(ids)
        for i, block in enumerate(self.blocks):
            x = block(x, pos, allow[:, None], caches[i] if caches is not None else None)
        return self.head(self.norm(x))

count = lambda net: sum(p.numel() for p in net.parameters())
print(f"파라미터 {count(LanguageModel()):,}")

In [ ]:
def fit(net, X, lr=0.003, batch=16, seed=0):
    """6장과 같다: 1에폭, AdamW, 처음 10% 걸음 동안 올리고 코사인으로 줄인다."""
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.0)
    shuffle = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed))
    steps = math.ceil(len(X) / batch)
    warm = max(1, steps // 10)
    net.train()
    for step in range(1, steps + 1):
        ids = X[shuffle[(step - 1) * batch:step * batch]]
        for group in optimizer.param_groups:
            group["lr"] = lr * min(1, step / warm) * 0.5 * (1 + math.cos(math.pi * step / steps))
        out = net(ids[:, :-1])
        loss = F.cross_entropy(out.reshape(-1, VOCAB), ids[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    net.eval()
    return net

def loss_of(net, X):
    with torch.inference_mode():
        return sum(F.cross_entropy(net(X[k:k + 8, :-1]).reshape(-1, VOCAB), X[k:k + 8, 1:].reshape(-1), reduction="sum").item()
                   for k in range(0, len(X), 8)) / X[:, 1:].numel()

torch.manual_seed(0)
lap = time.perf_counter()
net = fit(LanguageModel(), train_rows[:16_000_000 // LENGTH])     # 7장과 같은 토큰 1,600만 개, 같은 시드
print(f"학습 {(time.perf_counter() - lap) / 60:.1f}분, 검증 손실 {loss_of(net, val_rows[:200]):.4f}")

In [ ]:
# 양자화할 가중치: 토큰 표, 블록의 선형층 가중치, 출력층. 편향과 층 정규화는 작아서 32비트로 둔다
mats = {n: p for n, p in net.named_parameters() if p.dim() == 2}
part = lambda n: ("토큰 표" if n.startswith("table") else "출력층" if n.startswith("head")
                  else "어텐션" if ".attend." in n else "MLP")
total = sum(p.numel() for p in net.parameters())
print(f"전체 파라미터 {total:,}, 그 가운데 행렬 {sum(p.numel() for p in mats.values()):,}({sum(p.numel() for p in mats.values()) / total:.1%})")
for k in ["토큰 표", "어텐션", "MLP", "출력층"]:
    ps = [p for n, p in mats.items() if part(n) == k]
    print(f"{k:<4} 행렬 {len(ps):>2}개, 값 {sum(p.numel() for p in ps):>9,}개")
# 텐서마다 '가장 큰 절댓값이 표준편차의 몇 배인가'(튀는 값의 정도)
ratio = {n: (p.abs().max() / p.std()).item() for n, p in mats.items()}
for n in sorted(ratio, key=ratio.get)[::-1][:3] + sorted(ratio, key=ratio.get)[:2]:
    print(f"{n:<28} 최대/표준편차 {ratio[n]:5.1f}  최댓값 {mats[n].abs().max():.3f}")

## 8.2 정수로 바꾸기: 배율과 묶음

In [ ]:
def quantize(w, bits, group):
    """대칭 양자화. 묶음마다 배율 = 절댓값의 최대 / (2^(bits-1) - 1), 정수 = round(값 / 배율).
    group: "tensor"(텐서 전체에 배율 하나), "row"(출력 하나의 행마다), 정수(입력 방향으로 그 개수씩 묶음)."""
    qmax = 2 ** (bits - 1) - 1
    g = w.reshape(1, -1) if group == "tensor" else w.reshape(w.shape[0], -1) if group == "row" else w.reshape(-1, group)
    scale = (g.abs().amax(dim=1, keepdim=True) / qmax).clamp(min=1e-12)
    q = torch.round(g / scale).clamp(-qmax, qmax)
    return (q * scale).reshape(w.shape), q, scale

rel_error = lambda w, back: ((w - back).norm() / w.norm()).item()     # 상대 오차 ‖w − ŵ‖ / ‖w‖

# 예: 튀는 값이 가장 큰 행렬 하나를 4비트로 바꿀 때, 배율 하나의 범위에 따라 0이 되어 버리는 값의 비율
name = max(ratio, key=ratio.get)
w = mats[name].detach()
for group in ["tensor", "row", 64, 32]:
    back, q, scale = quantize(w, 4, group)
    print(f"{name} 4비트 {str(group):>6}: 상대 오차 {rel_error(w, back):.4f}, 0이 된 값 {(q == 0).float().mean():.1%}, 배율 {scale.numel():,}개")

BITS, GROUPS = [8, 6, 4, 3, 2], ["tensor", "row", 64, 32]
err = {}
for bits in BITS:
    for group in GROUPS:
        err[bits, group] = float(np.mean([rel_error(p.detach(), quantize(p.detach(), bits, group)[0]) for p in mats.values()]))
    print(f"{bits}비트  " + "  ".join(f"{str(g):>6} {err[bits, g]:.4f}" for g in GROUPS))

## 8.3 손실과 다음 토큰으로 재기

In [ ]:
import copy
def quantized(net, bits, group, parts=("토큰 표", "어텐션", "MLP", "출력층")):
    """가중치를 양자화했다가 되돌린 값으로 바꾼 모델의 복사본(계산은 32비트로 하되, 값은 정수가 나타낼 수 있는 것뿐)."""
    q = copy.deepcopy(net)
    with torch.no_grad():
        for n, p in q.named_parameters():
            if p.dim() == 2 and part(n) in parts:
                p.copy_(quantize(p, bits, group)[0])
    return q

VAL = val_rows[:100]                                       # 검증용 줄 100개(맞힐 토큰 51,200개)
def judge(model):
    """검증 손실, 32비트 모델과 다음 토큰(1등)이 같은 비율, 32비트 모델과의 KL 발산(토큰당 평균).
    로짓이 크므로(토큰마다 8,000개) 10줄씩 나눠 두 모델을 함께 계산한다."""
    loss = same = kl = 0.0
    n = VAL[:, 1:].numel()
    with torch.inference_mode():
        for k in range(0, len(VAL), 10):
            x, y = VAL[k:k + 10, :-1], VAL[k:k + 10, 1:]
            lb, lq = F.log_softmax(net(x), dim=-1), F.log_softmax(model(x), dim=-1)
            loss += F.nll_loss(lq.reshape(-1, VOCAB), y.reshape(-1), reduction="sum").item()
            same += (lq.argmax(dim=-1) == lb.argmax(dim=-1)).sum().item()
            kl += (lb.exp() * (lb - lq)).sum().item()
    return {"loss": loss / n, "same": same / n, "kl": kl / n}

base_loss = judge(net)["loss"]
print(f"32비트: 검증 손실 {base_loss:.4f}")
CONFIGS = [(8, "tensor"), (8, "row"), (6, "row"), (4, "tensor"), (4, "row"), (4, 64), (4, 32), (3, "row"), (3, 32), (2, 32)]
qual = {}
for bits, group in CONFIGS:
    qual[bits, group] = judge(quantized(net, bits, group))
    r = qual[bits, group]
    print(f"{bits}비트 {str(group):>6}: 가중치 오차 {err[bits, group]:.4f}  검증 손실 {r['loss']:.4f}({r['loss'] - base_loss:+.4f})  "
          f"1등 토큰 같음 {r['same']:.1%}  KL {r['kl']:.4f}", flush=True)

## 8.4 어느 부분이 예민한가

In [ ]:
# 한 부분만 3비트(행마다 배율)로 바꾸고 나머지는 32비트로 둔다
sens = {}
for k in ["토큰 표", "어텐션", "MLP", "출력층"]:
    ws = [p.detach() for n, p in mats.items() if part(n) == k]
    e = float(np.mean([rel_error(w, quantize(w, 3, "row")[0]) for w in ws]))
    sens[k] = (e, judge(quantized(net, 3, "row", parts=(k,))))
    print(f"{k:<4}만 3비트: 가중치 오차 {e:.4f}  검증 손실 {sens[k][1]['loss'] - base_loss:+.4f}  1등 토큰 같음 {sens[k][1]['same']:.1%}  "
          f"KL {sens[k][1]['kl']:.4f}", flush=True)
# 출력층만 8비트로 두고 나머지를 3비트로 바꾸면
mixed = quantized(quantized(net, 3, "row", parts=("토큰 표", "어텐션", "MLP")), 8, "row", parts=("출력층",))
qual["섞음"] = judge(mixed)
print(f"출력층 8비트 + 나머지 3비트: 검증 손실 {qual['섞음']['loss'] - base_loss:+.4f}  1등 토큰 같음 {qual['섞음']['same']:.1%}  KL {qual['섞음']['kl']:.4f}")

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter, NullFormatter
for path in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf"))
             + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(path)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 9, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"

fig, axes = plt.subplots(2, 1, figsize=(5.2, 6.0), dpi=260, gridspec_kw={"height_ratios": [1, 1.1]})
ax = axes[0]
w = mats[name].detach().flatten()
ax.hist(w.numpy(), bins=200, color=GRAY)
for v in torch.unique(quantize(mats[name].detach(), 4, "tensor")[0]):     # 텐서 전체에 배율 하나를 쓴 4비트가 나타낼 수 있는 값
    ax.axvline(v.item(), color=RED, lw=0.7, alpha=0.8)
ax.set_yscale("log"); ax.set_xlabel(f"{name}의 가중치 값"); ax.set_ylabel("값의 수(로그 눈금)")
ax.set_title(f"① 튀는 값이 가장 큰 행렬. 빨간 선은 텐서 전체에 배율 하나를 쓴\n"
             f"4비트 값 15개 가운데 실제로 쓰인 {len(torch.unique(quantize(mats[name].detach(), 4, 'tensor')[0]))}개",
             loc="left", fontsize=8.4, color="#374151")
ax = axes[1]
for group, color, label in [("tensor", RED, "텐서 전체"), ("row", ORANGE, "행마다"), (64, BLUE, "64개 묶음"), (32, GREEN, "32개 묶음")]:
    ax.plot(BITS, [err[b, group] for b in BITS], "-o", color=color, ms=4, lw=1.3, label=f"배율: {label}")
ax.set_yscale("log"); ax.set_xticks(BITS); ax.invert_xaxis()
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlabel("비트 수"); ax.set_ylabel("상대 오차(행렬 평균, 로그 눈금)")
ax.set_title("② 비트 수와 배율의 범위에 따른 가중치 오차", loc="left", fontsize=8.4, color="#374151")
ax.legend(frameon=False, fontsize=8)
for a in axes:
    a.grid(color="#eef0f3", lw=0.6); a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(h_pad=1.2); fig.savefig("fig_8_error.png"); plt.close(fig)

In [ ]:
GCOLOR = {"tensor": RED, "row": ORANGE, 64: BLUE, 32: GREEN}
GNAME = {"tensor": "텐서", "row": "행", 64: "64묶음", 32: "32묶음"}
keys = [k for k in qual if k != "섞음"]
labels = [f"{b}비트·{GNAME[g]}" for b, g in keys]
fig, axes = plt.subplots(1, 3, figsize=(5.2, 4.2), dpi=260, sharey=True)
y = np.arange(len(keys))[::-1]
cols = [GCOLOR[g] for _, g in keys]
panels = [("① 가중치 상대 오차", [err[k] for k in keys], "log"),
          ("② 검증 손실의 증가", [qual[k]["loss"] - base_loss for k in keys], "symlog"),
          ("③ 1등 토큰 같음(%)", [qual[k]["same"] * 100 for k in keys], "linear")]
for ax, (title, vals, scale) in zip(axes, panels):
    ax.scatter(vals, y, c=cols, s=20, zorder=3)
    if scale == "symlog":
        ax.set_xscale("symlog", linthresh=0.01); ax.set_xlim(-0.002, max(vals) * 4); ax.set_xticks([0, 0.01, 0.1, 1])
    elif scale == "log":
        ax.set_xscale("log"); ax.set_xlim(min(vals) * 0.6, max(vals) * 3); ax.set_xticks([0.01, 0.1, 1])
    else:
        ax.set_xlim(0, 108); ax.set_xticks([0, 50, 100])
    for yi, v in zip(y, vals):
        frac = ax.transLimits.transform(ax.transScale.transform((v, 0)))[0]     # 점이 축의 어디쯤에 있는지(0~1)
        right = frac > 0.55                                                     # 오른쪽에 있는 점은 글자를 왼쪽에 쓴다
        ax.annotate(f"{v:.4f}" if scale == "symlog" else f"{v:.3f}" if scale == "log" else f"{v:.1f}", (v, yi),
                    xytext=(-5 if right else 5, 0), textcoords="offset points", ha="right" if right else "left",
                    va="center", fontsize=8, color="#4b5563")
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    ax.set_title(title, loc="left", fontsize=8.4, color="#374151")
    ax.grid(axis="x", color="#eef0f3", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", labelsize=8)
axes[0].set_yticks(y); axes[0].set_yticklabels(labels, fontsize=8)
for t, c in zip(axes[0].get_yticklabels(), cols):
    t.set_color(c)
fig.tight_layout(w_pad=0.6); fig.savefig("fig_8_quality.png"); plt.close(fig)

fig, ax = plt.subplots(figsize=(5.2, 2.2), dpi=260)
names = list(sens)
inc = [sens[k][1]["loss"] - base_loss for k in names]
ax.barh(names, inc, color=[GRAY, BLUE, GREEN, ORANGE], height=0.6)
for i, k in enumerate(names):
    ax.text(inc[i], i, f"  +{inc[i]:.4f}  (가중치 오차 {sens[k][0]:.3f}, 1등 같음 {sens[k][1]['same']:.1%})", va="center", fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("검증 손실의 증가"); ax.set_xlim(0, max(inc) * 2.9)
ax.tick_params(labelsize=8)
ax.grid(axis="x", color="#eef0f3", lw=0.6); ax.set_axisbelow(True); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_8_parts.png"); plt.close(fig)

## 8.5 메모리와 속도

In [ ]:
def pack4(q):
    """4비트 정수(−7~7) 두 개를 바이트 하나에 담는다: 8을 더해 0~15로 옮긴 뒤 앞 값은 아래 4비트, 뒤 값은 위 4비트."""
    u = (q.flatten().to(torch.int16) + 8).to(torch.uint8)
    return u[0::2] | (u[1::2] << 4)
def unpack4(b, shape):
    u = torch.stack([b & 15, b >> 4], dim=1).flatten().to(torch.int16) - 8
    return u.reshape(shape).float()

store = {}
for bits, group in [(8, "row"), (4, 32)]:
    nbytes, same = 0, True
    for n, p in mats.items():
        _, q, scale = quantize(p.detach(), bits, group)
        if bits == 8:
            data = q.to(torch.int8)
        else:
            data = pack4(q); same &= torch.equal(unpack4(data, q.shape), q)
        nbytes += data.numel() * data.element_size() + scale.numel() * 2      # 배율은 16비트로 저장한다고 센다
    rest = sum(p.numel() * 4 for p in net.parameters() if p.dim() != 2)
    store[bits] = nbytes + rest
    print(f"{bits}비트({group}): 행렬 {nbytes:,}바이트 + 나머지 {rest:,}바이트 = {store[bits]:,}바이트, 풀었을 때 같음 {same}")
store[32] = sum(p.numel() * 4 for p in net.parameters())
print(f"32비트: {store[32]:,}바이트.  8비트는 {store[8] / store[32]:.1%}, 4비트는 {store[4] / store[32]:.1%}")

In [ ]:
class QuantLinear(nn.Module):
    """가중치를 8비트 정수와 배율로 저장하고, 부를 때마다 32비트로 되돌려 곱한다."""
    def __init__(self, linear, bits=8):
        super().__init__()
        _, q, scale = quantize(linear.weight.detach(), bits, "row")
        self.register_buffer("q", q.to(torch.int8)); self.register_buffer("scale", scale)
        self.bias = linear.bias
    def forward(self, x):
        return F.linear(x, self.q.float() * self.scale, self.bias)

def swap(model):
    m = copy.deepcopy(model)
    for block in m.blocks:
        a = block.attend
        a.to_q, a.to_k, a.to_v, a.out = (QuantLinear(l) for l in (a.to_q, a.to_k, a.to_v, a.out))
        block.mlp[0], block.mlp[2] = QuantLinear(block.mlp[0]), QuantLinear(block.mlp[2])
    m.head = QuantLinear(m.head)
    return m

def median_time(fn, repeat=7):
    fn()
    laps = []
    for _ in range(repeat):
        lap = time.perf_counter(); fn(); laps.append(time.perf_counter() - lap)
    return float(np.median(laps))

int8_net = swap(net)
speed = {}
with torch.inference_mode():
    print(f"8비트 저장 모델과, 같은 값을 되돌려 32비트로 계산한 모델의 로짓 차이 최대 {(int8_net(VAL[:2, :64]) - quantized(net, 8, 'row', parts=('어텐션', 'MLP', '출력층'))(VAL[:2, :64])).abs().max():.2e}")
    for label, model in [("32비트", net), ("8비트 저장", int8_net)]:
        for B in [1, 32]:
            caches = [{} for _ in model.blocks]
            model(val_rows[:B, :448], caches=caches)
            one = val_rows[:B, 448:449]
            def step():
                for c in caches:
                    c["k"], c["v"] = c["k"][:, :, :448], c["v"][:, :, :448]
                caches[0]["real"] = caches[0]["real"][:, :448]
                model(one, caches=caches)
            speed[label, B] = median_time(step)
        speed[label, "prefill"] = median_time(lambda: model(val_rows[:1, :448], caches=[{} for _ in model.blocks]))
        print(f"{label}: 디코드 한 걸음 배치 1 {speed[label, 1] * 1000:.2f}ms, 배치 32 {speed[label, 32] * 1000:.2f}ms, "
              f"프리필 448토큰 {speed[label, 'prefill'] * 1000:.1f}ms", flush=True)
    W = net.head.weight
    q8, s8 = int8_net.head.q, int8_net.head.scale
    t_back = median_time(lambda: q8.float() * s8, repeat=21)
    x1 = torch.randn(1, 192, generator=torch.Generator().manual_seed(0))
    t_mul = median_time(lambda: F.linear(x1, W), repeat=21)
    print(f"출력층 하나: 8비트를 32비트로 되돌리기 {t_back * 1000:.3f}ms, 토큰 하나의 곱셈 {t_mul * 1000:.3f}ms")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(5.2, 2.9), dpi=260, gridspec_kw={"width_ratios": [1, 1.25]})
ax = axes[0]
labels = ["32비트", "8비트\n(행마다)", "4비트\n(32개 묶음)"]
vals = [store[32] / 2**20, store[8] / 2**20, store[4] / 2**20]
ax.bar(labels, vals, color=[GRAY, BLUE, GREEN], width=0.6)
for i, v in enumerate(vals):
    ax.text(i, v + 0.3, f"{v:.1f}", ha="center", fontsize=8)
ax.set_ylabel("모델을 담는 메모리(MiB)", fontsize=8.4); ax.tick_params(labelsize=8)
ax.set_title("① 메모리", loc="left", fontsize=8.4, color="#374151")
ax = axes[1]
keys = [(1, "디코드\n배치 1"), (32, "디코드\n배치 32"), ("prefill", "프리필\n448토큰")]
x = np.arange(len(keys))
ax.bar(x - 0.2, [speed["32비트", k] * 1000 for k, _ in keys], 0.38, color=GRAY, label="32비트")
ax.bar(x + 0.2, [speed["8비트 저장", k] * 1000 for k, _ in keys], 0.38, color=BLUE, label="8비트로 저장,\n부를 때 되돌림")
ax.set_xticks(x); ax.set_xticklabels([l for _, l in keys]); ax.set_ylabel("시간(ms)", fontsize=8.4); ax.tick_params(labelsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title("② 속도", loc="left", fontsize=8.4, color="#374151")
for a in axes:
    a.grid(axis="y", color="#eef0f3", lw=0.6); a.set_axisbelow(True); a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_8_memory.png"); plt.close(fig)